# P2 - Electricity Load Forecasting (RNNs)
## Deep Learning — Master's Degree in Artificial Intelligence (2025–2026)

**Authors:**
- Patricia Guadalupe Alvarenga Mairena
- Francisco Manuel Vázquez Fernández

## 1. Setup and Imports

We import all necessary libraries and set random seeds for reproducibility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")

---
## 2. Load and Explore Dataset

We load the NYISO hourly electricity load dataset and perform an initial exploration: check shape, columns, data types, missing values, and visualize the total load over time.

In [ ]:
# Load the dataset
df = pd.read_csv("nyiso_hourly_load.csv", parse_dates=["Time Stamp"])

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

In [ ]:
# Visualize total load over time
plt.figure(figsize=(14, 4))
plt.plot(df["Time Stamp"], df["total_load"], linewidth=0.5)
plt.title("Total Electricity Load — New York State (2021–2025)")
plt.xlabel("Date")
plt.ylabel("Load (MW)")
plt.tight_layout()
plt.show()

In [ ]:
# Basic statistics
df.describe()

---
## 3. Data Splitting (Train / Validation / Test)

We split the data temporally:
- **Training:** 2021–2023
- **Validation:** 2024
- **Test:** 2025

The `Time Stamp` column is used only for splitting and then discarded from model inputs. The input features are the 11 zone loads plus the total load (12 features).

In [ ]:
# Temporal split
train_df = df[df["Time Stamp"].dt.year <= 2023].copy()
val_df   = df[df["Time Stamp"].dt.year == 2024].copy()
test_df  = df[df["Time Stamp"].dt.year == 2025].copy()

print(f"Train: {train_df.shape}  ({train_df['Time Stamp'].min()} — {train_df['Time Stamp'].max()})")
print(f"Val:   {val_df.shape}  ({val_df['Time Stamp'].min()} — {val_df['Time Stamp'].max()})")
print(f"Test:  {test_df.shape}  ({test_df['Time Stamp'].min()} — {test_df['Time Stamp'].max()})")

# Drop Time Stamp — keep only numeric features
feature_columns = [c for c in df.columns if c != "Time Stamp"]
print(f"\nFeature columns ({len(feature_columns)}): {feature_columns}")

train_data = train_df[feature_columns].values
val_data   = val_df[feature_columns].values
test_data  = test_df[feature_columns].values

---
## 4. Normalization

We apply z-score normalization using statistics computed **only on the training set**. We store the mean and std of the `total_load` column to be able to denormalize predictions later for reporting MAE in original units (MW).

In [ ]:
# Compute normalization statistics from training set
train_mean = train_data.mean(axis=0)
train_std  = train_data.std(axis=0)

# Index of total_load column (last column)
total_load_idx = feature_columns.index("total_load")
total_load_mean = train_mean[total_load_idx]
total_load_std  = train_std[total_load_idx]

print(f"total_load — mean: {total_load_mean:.2f} MW, std: {total_load_std:.2f} MW")

# Apply normalization
train_norm = (train_data - train_mean) / train_std
val_norm   = (val_data   - train_mean) / train_std
test_norm  = (test_data  - train_mean) / train_std

---
## 5. Sequence Generation (Windowing)

We transform the data into supervised learning samples using a sliding window approach.

- **Input:** a window of `SEQ_LEN` consecutive hourly observations (all 12 features).
- **Target:** the `total_load` value `HORIZON` hours after the last observation in the window.

We choose `SEQ_LEN = 48` (2 days of context) and `HORIZON = 3` (predict 3 hours ahead).

We also define a helper function to compute **denormalized MAE** — the evaluation metric required for the practice.

In [ ]:
SEQ_LEN = 48   # 48 hours of past observations
HORIZON = 3    # Predict 3 hours ahead

def create_sequences(data, seq_len, horizon, target_idx):
    """
    Create input/output sequences from a 2D array.
    
    Parameters:
        data: np.array of shape (n_timesteps, n_features)
        seq_len: number of past time steps to use as input
        horizon: how many steps ahead to predict
        target_idx: column index of the target variable
    
    Returns:
        X: np.array of shape (n_samples, seq_len, n_features)
        y: np.array of shape (n_samples,)
    """
    X, y = [], []
    for i in range(len(data) - seq_len - horizon + 1):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len + horizon - 1, target_idx])
    return np.array(X), np.array(y)

# Generate sequences
X_train, y_train = create_sequences(train_norm, SEQ_LEN, HORIZON, total_load_idx)
X_val,   y_val   = create_sequences(val_norm,   SEQ_LEN, HORIZON, total_load_idx)
X_test,  y_test  = create_sequences(test_norm,  SEQ_LEN, HORIZON, total_load_idx)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

In [ ]:
def denormalized_mae(y_true_norm, y_pred_norm, mean, std):
    """Compute MAE in original scale (MW)."""
    y_true = y_true_norm * std + mean
    y_pred = y_pred_norm * std + mean
    return np.mean(np.abs(y_true - y_pred))

# Dictionary to store results for final comparison
results = {}

---
# Part 1: Baselines

## 6. Baseline 1 — Last Value

The simplest baseline: predict the total load 3 hours ahead using the **last observed total load** value in the input window. This is equivalent to assuming the load stays constant.

In [ ]:
# Last-value baseline: use the last total_load in the window as the prediction
y_pred_last = X_test[:, -1, total_load_idx]

mae_last = denormalized_mae(y_test, y_pred_last, total_load_mean, total_load_std)
results["Last Value"] = mae_last
print(f"Last-Value Baseline — Denormalized MAE: {mae_last:.2f} MW")

## 7. Baseline 2 — Daily (Same Hour Previous Day)

We predict using the total load from **24 hours before the target time**. Since the target is 3 hours after the end of the window, we look back `24 - 3 = 21` steps from the end of the window (i.e., position `-21` in the sequence).

This baseline captures the daily seasonality of electricity demand.

In [ ]:
# Daily baseline: value from the same hour one day earlier
# Target is at position (end_of_window + HORIZON), i.e., 3 hours after the last step.
# Same hour previous day = 24h before the target = window[-1] - 24 + HORIZON = window[-(24 - HORIZON + 1)]
# Since HORIZON=3: position is -(24 - 3) = -21 from the end, i.e., index SEQ_LEN - 21

offset = 24 - HORIZON  # 21 steps back from end of window
y_pred_daily = X_test[:, -offset, total_load_idx]

mae_daily = denormalized_mae(y_test, y_pred_daily, total_load_mean, total_load_std)
results["Daily"] = mae_daily
print(f"Daily Baseline — Denormalized MAE: {mae_daily:.2f} MW")

## 8. Neural Baseline — Conv1D + Dense

We build a neural baseline combining **1D convolutional layers** (to extract local temporal patterns) with **Dense layers**. This serves as a non-recurrent neural reference point for comparison with the RNN models.

In [ ]:
# Common training hyperparameters (shared across all neural models)
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 1e-3

def plot_training_history(history, title=""):
    """Plot training and validation loss curves."""
    plt.figure(figsize=(8, 3))
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MSE)")
    plt.title(f"Training Curves — {title}")
    plt.legend()
    plt.tight_layout()
    plt.show()

def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate a Keras model and store denormalized MAE."""
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = denormalized_mae(y_test, y_pred, total_load_mean, total_load_std)
    results[model_name] = mae
    print(f"{model_name} — Denormalized MAE: {mae:.2f} MW")
    return mae

In [ ]:
# Neural baseline: Conv1D + Dense
n_features = X_train.shape[2]

model_cnn = keras.Sequential([
    layers.Conv1D(filters=32, kernel_size=3, activation="relu",
                  input_shape=(SEQ_LEN, n_features)),
    layers.Conv1D(filters=32, kernel_size=3, activation="relu"),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)
], name="CNN_Baseline")

model_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

model_cnn.summary()

In [ ]:
# Train CNN baseline
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history_cnn = model_cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
plot_training_history(history_cnn, "Conv1D + Dense Baseline")
evaluate_model(model_cnn, X_test, y_test, "Conv1D + Dense")

---
# Part 2: Recurrent Models

## 9. SimpleRNN Model

We build a model using `SimpleRNN` layers. We will experiment with:
- Number of units (e.g., 32, 64)
- Number of stacked RNN layers (1 vs 2)
- Dropout for regularization

We keep the same optimizer (Adam), learning rate, batch size and early stopping strategy as the CNN baseline for fair comparison.

In [ ]:
# --- Experiment 1: SimpleRNN with 1 layer, 64 units ---
model_srnn_v1 = keras.Sequential([
    layers.SimpleRNN(64, input_shape=(SEQ_LEN, n_features)),
    layers.Dense(1)
], name="SimpleRNN_v1")

model_srnn_v1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

history_srnn_v1 = model_srnn_v1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=1
)

plot_training_history(history_srnn_v1, "SimpleRNN v1 (64 units, 1 layer)")
evaluate_model(model_srnn_v1, X_test, y_test, "SimpleRNN v1")

In [ ]:
# --- Experiment 2: SimpleRNN with 2 stacked layers + dropout ---
model_srnn_v2 = keras.Sequential([
    layers.SimpleRNN(64, return_sequences=True, input_shape=(SEQ_LEN, n_features)),
    layers.Dropout(0.2),
    layers.SimpleRNN(32),
    layers.Dropout(0.2),
    layers.Dense(1)
], name="SimpleRNN_v2")

model_srnn_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

history_srnn_v2 = model_srnn_v2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=1
)

plot_training_history(history_srnn_v2, "SimpleRNN v2 (2 layers + dropout)")
evaluate_model(model_srnn_v2, X_test, y_test, "SimpleRNN v2")

**SimpleRNN — Discussion:**

*TODO: Compare v1 vs v2 results. Discuss which configuration works better and why. Comment on the vanishing gradient problem that SimpleRNN is known for, especially with longer sequences. Justify the final choice.*

Selected final model: **SimpleRNN v?**

---
## 10. LSTM Model

We build a model using `LSTM` layers. LSTMs address the vanishing gradient problem through gating mechanisms, which should help with our 48-step input sequences. We explore similar design choices as with the SimpleRNN.

In [ ]:
# --- Experiment 1: LSTM with 1 layer, 64 units ---
model_lstm_v1 = keras.Sequential([
    layers.LSTM(64, input_shape=(SEQ_LEN, n_features)),
    layers.Dense(1)
], name="LSTM_v1")

model_lstm_v1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

history_lstm_v1 = model_lstm_v1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=1
)

plot_training_history(history_lstm_v1, "LSTM v1 (64 units, 1 layer)")
evaluate_model(model_lstm_v1, X_test, y_test, "LSTM v1")

In [ ]:
# --- Experiment 2: LSTM with 2 stacked layers + dropout ---
model_lstm_v2 = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, n_features)),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(1)
], name="LSTM_v2")

model_lstm_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

history_lstm_v2 = model_lstm_v2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=1
)

plot_training_history(history_lstm_v2, "LSTM v2 (2 layers + dropout)")
evaluate_model(model_lstm_v2, X_test, y_test, "LSTM v2")

**LSTM — Discussion:**

*TODO: Compare v1 vs v2 results. Discuss how LSTM handles long-term dependencies compared to SimpleRNN. Comment on training time vs performance trade-off. Justify final choice.*

Selected final model: **LSTM v?**

---
## 11. GRU Model

We build a model using `GRU` layers. GRUs are similar to LSTMs but with a simpler gating mechanism (2 gates instead of 3), which often results in faster training while achieving comparable performance.

In [ ]:
# --- Experiment 1: GRU with 1 layer, 64 units ---
model_gru_v1 = keras.Sequential([
    layers.GRU(64, input_shape=(SEQ_LEN, n_features)),
    layers.Dense(1)
], name="GRU_v1")

model_gru_v1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

history_gru_v1 = model_gru_v1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=1
)

plot_training_history(history_gru_v1, "GRU v1 (64 units, 1 layer)")
evaluate_model(model_gru_v1, X_test, y_test, "GRU v1")

In [ ]:
# --- Experiment 2: GRU with 2 stacked layers + dropout ---
model_gru_v2 = keras.Sequential([
    layers.GRU(64, return_sequences=True, input_shape=(SEQ_LEN, n_features)),
    layers.Dropout(0.2),
    layers.GRU(32),
    layers.Dropout(0.2),
    layers.Dense(1)
], name="GRU_v2")

model_gru_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

history_gru_v2 = model_gru_v2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=1
)

plot_training_history(history_gru_v2, "GRU v2 (2 layers + dropout)")
evaluate_model(model_gru_v2, X_test, y_test, "GRU v2")

**GRU — Discussion:**

*TODO: Compare v1 vs v2 results. Discuss GRU vs LSTM trade-offs: fewer parameters, faster training, comparable performance. Justify final choice.*

Selected final model: **GRU v?**

---
## 12. Results Comparison and Final Analysis

We summarize the denormalized MAE (MW) on the test set for all models and provide a comparative analysis.

In [ ]:
# Summary table
results_df = pd.DataFrame(
    list(results.items()),
    columns=["Model", "Denormalized MAE (MW)"]
).sort_values("Denormalized MAE (MW)")

print("=" * 50)
print("FINAL RESULTS — Test Set (Denormalized MAE in MW)")
print("=" * 50)
print(results_df.to_string(index=False))
print("=" * 50)

In [ ]:
# Bar chart comparison
plt.figure(figsize=(10, 5))
colors = ["#aaaaaa", "#aaaaaa", "#5599cc", "#dd7744", "#dd7744", "#44aa77", "#44aa77", "#cc66aa", "#cc66aa"]
bars = plt.barh(results_df["Model"], results_df["Denormalized MAE (MW)"],
                color=colors[:len(results_df)])
plt.xlabel("Denormalized MAE (MW)")
plt.title("Model Comparison — Test Set Performance")
plt.gca().invert_yaxis()

# Add value labels
for bar in bars:
    width = bar.get_width()
    plt.text(width + 20, bar.get_y() + bar.get_height() / 2,
             f"{width:.0f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

### Final Discussion

*TODO: Fill in after running all experiments. Points to cover:*

1. **Baselines vs Neural models:** Did the neural models improve over the simple baselines? By how much?

2. **Recurrent vs Non-recurrent:** How do the RNN-based models compare to the Conv1D + Dense baseline? Is the sequential modeling capability of RNNs beneficial for this task?

3. **SimpleRNN vs LSTM vs GRU:**
   - Which recurrent architecture achieved the best MAE?
   - How do training times compare?
   - Did stacking layers / adding dropout help?

4. **Daily baseline relevance:** Electricity load has strong daily seasonality. How competitive is the daily baseline compared to more complex models?

5. **Limitations and future work:**
   - We did not include time-based features (hour of day, day of week) which could improve performance.
   - A larger hyperparameter search could yield better results.
   - Attention mechanisms or Transformer-based models could be explored.